# Atividade — Meios de Transporte

**Base de dados:** `Tabela_13_Meio_de_transporte.xlsx`

Esta atividade utiliza as abas `Exemplo gráfico Brasil` e `BR GR UF MU`. Na segunda parte, o valor geral é obtido pela média dos percentuais das três categorias de cor/raça disponíveis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
# Localiza a planilha na mesma pasta do notebook
pasta_projeto = Path.cwd()
arquivo_dados = pasta_projeto / 'Tabela_13_Meio_de_transporte.xlsx'

print(f'Arquivo utilizado: {arquivo_dados.name}')


## PARTE 1 — Distribuição por cor/raça

In [ ]:
# Importação da tabela utilizada nos gráficos
brasil = pd.read_excel(
    arquivo_dados,
    sheet_name='Exemplo gráfico Brasil',
    header=1
)

brasil.columns = ['Meio de transporte', 'Branca', 'Preta ou Parda', 'Indígena']
brasil


In [ ]:
# Criação dos três gráficos de pizza
categorias = ['Branca', 'Preta ou Parda', 'Indígena']
fig, eixos = plt.subplots(1, 3, figsize=(17, 5.8))

for eixo, categoria in zip(eixos, categorias):
    valores = brasil[categoria]
    eixo.pie(
        valores,
        labels=brasil['Meio de transporte'],
        autopct='%.2f%%',
        startangle=90,
        wedgeprops={'linewidth': 1, 'edgecolor': 'white'}
    )
    eixo.set_title(categoria)

fig.suptitle('Meios de transporte das mulheres por cor/raça — Brasil, 2022', fontsize=14)
plt.tight_layout()
plt.savefig('distribuicao_transportes.png', dpi=300, bbox_inches='tight')
plt.show()


## Comparação dos percentuais

As diferenças são apresentadas em **pontos percentuais (p.p.)**, obtidas pela subtração entre os percentuais de cada categoria.

- **A pé:** Brancas − Pretas/Pardas = **−5,4 p.p.**; Brancas − Indígenas = **−18,1 p.p.**; Pretas/Pardas − Indígenas = **−12,7 p.p.**
- **Bicicleta:** Brancas − Pretas/Pardas = **−2,0 p.p.**; Brancas − Indígenas = **−2,7 p.p.**; Pretas/Pardas − Indígenas = **−0,7 p.p.**
- **Motocicleta ou Mototáxi:** Brancas − Pretas/Pardas = **−4,3 p.p.**; Brancas − Indígenas = **−5,5 p.p.**; Pretas/Pardas − Indígenas = **−1,2 p.p.**
- **Automóvel, táxi ou assemelhados:** Brancas − Pretas/Pardas = **+21,2 p.p.**; Brancas − Indígenas = **+26,6 p.p.**; Pretas/Pardas − Indígenas = **+5,4 p.p.**
- **Transporte coletivo:** Brancas − Pretas/Pardas = **−9,4 p.p.**; Brancas − Indígenas = **+2,8 p.p.**; Pretas/Pardas − Indígenas = **+12,2 p.p.**
- **Outros:** Brancas − Pretas/Pardas = **−0,1 p.p.**; Brancas − Indígenas = **−3,1 p.p.**; Pretas/Pardas − Indígenas = **−3,0 p.p.**

Por exemplo, no automóvel, o percentual registrado para mulheres brancas é **21,2 pontos percentuais maior** que o de mulheres pretas ou pardas.

## PARTE 2 — Estados e municípios

In [ ]:
# Leitura da aba com informações de Brasil, estados e municípios
base_geografica = pd.read_excel(
    arquivo_dados,
    sheet_name='BR GR UF MU',
    header=None
)

ufs = [
    'Rondônia', 'Acre', 'Amazonas', 'Roraima', 'Pará', 'Amapá', 'Tocantins',
    'Maranhão', 'Piauí', 'Ceará', 'Rio Grande do Norte', 'Paraíba',
    'Pernambuco', 'Alagoas', 'Sergipe', 'Bahia', 'Minas Gerais',
    'Espírito Santo', 'Rio de Janeiro', 'São Paulo', 'Paraná',
    'Santa Catarina', 'Rio Grande do Sul', 'Mato Grosso do Sul',
    'Mato Grosso', 'Goiás', 'Distrito Federal'
]

somente_ufs = base_geografica[base_geografica[0].isin(ufs)].copy()
somente_ufs = somente_ufs.rename(columns={0: 'Estado'})
somente_ufs[['Estado']]


In [ ]:
# Padroniza os 18 campos de percentual: 3 categorias de raça x 6 transportes
meios = [
    'A pé', 'Bicicleta', 'Motocicleta ou Mototaxi',
    'Automóvel, taxi ou assemelhados', 'Transporte coletivo', 'Outros'
]
campos = ['Estado'] + [f'{grupo} - {meio}' for grupo in ['Branca', 'Preta ou Parda', 'Indígena'] for meio in meios]

somente_ufs = somente_ufs.iloc[:, :19].copy()
somente_ufs.columns = campos

for coluna in campos[1:]:
    somente_ufs[coluna] = pd.to_numeric(somente_ufs[coluna], errors='coerce')

long_ufs = somente_ufs.melt(
    id_vars='Estado',
    var_name='Categoria_Transporte',
    value_name='Percentual'
)

long_ufs[['Cor_Raca', 'Transporte']] = long_ufs['Categoria_Transporte'].str.split(' - ', n=1, expand=True)
long_ufs.head()


In [ ]:
# Média das três categorias de cor/raça para cada estado e transporte
resumo_ufs = (
    long_ufs.groupby(['Estado', 'Transporte'], as_index=False)['Percentual']
    .mean()
)

coletivo_ufs = resumo_ufs.query("Transporte == 'Transporte coletivo'").sort_values(
    'Percentual', ascending=False
)
coletivo_ufs


### Resultado — transporte coletivo

**Maior percentual médio:** **Rio de Janeiro — 52,23%**.

**Menor percentual médio:** **Rondônia — 4,33%**.

Os valores consideram a média simples das categorias Branca, Preta ou Parda e Indígena.

In [ ]:
# Seleção dos municípios a partir das linhas correspondentes à base municipal
municipios = base_geografica.iloc[36:, :19].copy()
municipios = municipios[municipios[0].notna()].copy()
municipios = municipios.rename(columns={0: 'Cidade'})
municipios.columns = campos
municipios = municipios.rename(columns={'Estado': 'Cidade'})

for coluna in municipios.columns[1:]:
    municipios[coluna] = pd.to_numeric(municipios[coluna], errors='coerce')

long_municipios = municipios.melt(
    id_vars='Cidade',
    var_name='Categoria_Transporte',
    value_name='Percentual'
)
long_municipios[['Cor_Raca', 'Transporte']] = long_municipios['Categoria_Transporte'].str.split(
    ' - ', n=1, expand=True
)
long_municipios.head()


In [ ]:
# Calcula a média por cidade e meio de transporte
resumo_cidades = (
    long_municipios.groupby(['Cidade', 'Transporte'], as_index=False)['Percentual']
    .mean()
)

top_bicicleta = (
    resumo_cidades[resumo_cidades['Transporte'] == 'Bicicleta']
    .dropna(subset=['Percentual'])
    .nlargest(1, 'Percentual')
)

top_automovel = (
    resumo_cidades[resumo_cidades['Transporte'] == 'Automóvel, taxi ou assemelhados']
    .dropna(subset=['Percentual'])
    .nlargest(1, 'Percentual')
)

print('Cidade com maior percentual médio de bicicleta:')
display(top_bicicleta)
print('Cidade com maior percentual médio de automóvel:')
display(top_automovel)


### Resultado — municípios

**Bicicleta:** **Tuiuti (SP) — 100,00%**.

**Automóvel:** **Bom Jesus do Oeste (SC) — 86,05%**.

Os resultados foram obtidos pela média dos percentuais das três categorias de cor/raça.

## Conclusão

A análise evidencia diferenças no uso dos meios de transporte entre as categorias de cor/raça. Na tabela nacional, o automóvel possui percentual mais elevado entre mulheres brancas, enquanto o transporte coletivo apresenta percentual mais elevado entre mulheres pretas ou pardas. Na escala municipal, os maiores percentuais médios identificados foram Tuiuti (SP), para bicicleta, e Bom Jesus do Oeste (SC), para automóvel.